In [2]:
!apt-get update -y -qq
!apt-get install -y -qq libgsl-dev build-essential
!git clone https://github.com/scwatts/fastspar.git
%cd fastspar
!./autogen.sh
!./configure
!make -j4
!./src/fastspar --version
print("FastSpar ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'fastspar'...
remote: Enumerating objects: 993, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 993 (delta 28), reused 37 (delta 18), pack-reused 936 (from 1)
Receiving objects: 100% (993/993), 766.16 KiB | 9.34 MiB/s, done.
Resolving deltas: 100% (654/654), done.
/content/fastspar/fastspar
checking for a BSD-compatible install... /usr/bin/install -c
checking whether build environment is sane... yes
checking for a race-free mkdir -p... /usr/bin/mkdir -p
checking for gawk... no
checking for mawk... mawk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking for g++... g++
checking whether the C++ compiler works... yes
checking for C++ compiler default output file name... a

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, shutil
import pandas as pd

GROUPS = ["IBD", "rCDI", "MetS", "MDR", "ICI"]

print("Copying count files from Drive → Colab\n")
for grp in GROUPS:
    src = f"/content/drive/MyDrive/{grp}_pod_network_counts.tsv"
    dst = f"/content/{grp}_pod_network_counts.tsv"

    if os.path.exists(src):
        shutil.copy(src, dst)
        df = pd.read_csv(dst, sep="\t", index_col=0)
        print(f"{grp}: copied — shape {df.shape}")
    else:
        print(f"{grp}: NOT FOUND at {src}")

print("\nReformatting to OTU-table layout (species as rows)...")
for grp in GROUPS:
    path = f"/content/{grp}_pod_network_counts.tsv"
    df = pd.read_csv(path, sep="\t", index_col=0)
    df = df.T
    df.reset_index(inplace=True)
    df.rename(columns={"index": "#OTU ID"}, inplace=True)
    df.to_csv(path, sep="\t", index=False)
    print(f"{grp}: reformatted — {df.shape}")

Copying count files from Drive → Colab

IBD: copied — shape (353, 95)
rCDI: copied — shape (127, 158)
MetS: copied — shape (317, 179)
MDR: copied — shape (102, 111)
ICI: copied — shape (220, 115)

Reformatting to OTU-table layout (species as rows)...
IBD: reformatted — (95, 354)
rCDI: reformatted — (158, 128)
MetS: reformatted — (179, 318)
MDR: reformatted — (111, 103)
ICI: reformatted — (115, 221)


In [5]:
import os, subprocess, glob, time

def run_full_fastspar(group, n_bootstrap=100, iterations=20):
    counts = f"/content/{group}_pod_network_counts.tsv"
    base_out = f"/content/fastspar/sparcc_{group}"
    boot_counts = f"{base_out}/bootstrap_counts"
    boot_cors = f"{base_out}/bootstrap_cors"
    boot_only = f"{base_out}/bootstrap_corr_only"

    os.makedirs(base_out, exist_ok=True)
    os.makedirs(boot_counts, exist_ok=True)
    os.makedirs(boot_cors, exist_ok=True)
    os.makedirs(boot_only, exist_ok=True)

    print(f"Processing: {group}")
    print("Step 1/4: Main correlations...")
    r = subprocess.run([
        "./src/fastspar", "--otu_table", counts,
        "--correlation", f"{base_out}/cor_{group}.csv",
        "--covariance", f"{base_out}/cov_{group}.csv",
        "--iterations", str(iterations), "--yes"
    ], cwd="/content/fastspar", capture_output=True, text=True)
    if r.returncode != 0:
        print("ERROR in main correlation:", r.stderr[:300]); return False
    print("Done")

    print(f"Step 2/4: Generating {n_bootstrap} bootstraps...")
    subprocess.run("rm -f /content/fastspar/boot_*.tsv", shell=True)
    r = subprocess.run([
        "./src/fastspar_bootstrap", "--otu_table", counts,
        "--number", str(n_bootstrap), "--prefix", "boot_"
    ], cwd="/content/fastspar", capture_output=True, text=True)
    if r.returncode != 0:
        print("ERROR in bootstrap:", r.stderr[:300]); return False

    boot_files = glob.glob("/content/fastspar/boot_*.tsv")
    for f in boot_files:
        shutil.move(f, boot_counts)
    boot_files = glob.glob(f"{boot_counts}/boot_*.tsv")
    print(f"{len(boot_files)} bootstrap files created")

    print(f"Step 3/4: Running bootstrap correlations ({len(boot_files)})...")
    for i, bf in enumerate(boot_files):
        base = os.path.splitext(os.path.basename(bf))[0]
        subprocess.run([
            "./src/fastspar", "--otu_table", bf,
            "--correlation", f"{boot_cors}/{base}.tsv",
            "--covariance", f"{boot_cors}/{base}_cov.tsv",
            "--iterations", str(iterations), "--yes"
        ], cwd="/content/fastspar", capture_output=True)
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(boot_files)}")
    print("Bootstrap correlations done")

    for f in glob.glob(f"{boot_cors}/*.tsv"):
        if "_cov" not in f:
            subprocess.run(["cp", f, boot_only])
    print(f"{len(os.listdir(boot_only))} correlation files ready")

    print("Step 4/4: Computing p-values...")
    r = subprocess.run([
        "./src/fastspar_pvalues", "--otu_table", counts,
        "--correlation", f"sparcc_{group}/cor_{group}.csv",
        "--prefix", f"sparcc_{group}/bootstrap_corr_only/boot_",
        "--permutations", str(n_bootstrap),
        "--outfile", f"sparcc_{group}/pvals_{group}.csv"
    ], cwd="/content/fastspar", capture_output=True, text=True)
    if r.returncode != 0:
        print("ERROR in p-values:", r.stderr[:300]); return False
    print("P-values computed successfully")
    return True

start = time.time()
for grp in GROUPS:
    success = run_full_fastspar(grp, n_bootstrap=100, iterations=20)
    print(f"{grp} {'COMPLETE' if success else 'FAILED'}")
print(f"\nTotal time: {(time.time() - start)/60:.1f} minutes")

Processing: IBD
Step 1/4: Main correlations...
Done
Step 2/4: Generating 100 bootstraps...
100 bootstrap files created
Step 3/4: Running bootstrap correlations (100)...
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 100/100
Bootstrap correlations done
100 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
IBD COMPLETE
Processing: rCDI
Step 1/4: Main correlations...
Done
Step 2/4: Generating 100 bootstraps...
100 bootstrap files created
Step 3/4: Running bootstrap correlations (100)...
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 100/100
Bootstrap correlations done
100 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
rCDI COMPLETE
Processing: Me

In [7]:
import shutil

print("VERIFICATION:")
for grp in GROUPS:
    cor_path = f"/content/fastspar/sparcc_{grp}/cor_{grp}.csv"
    pval_path = f"/content/fastspar/sparcc_{grp}/pvals_{grp}.csv"
    if os.path.exists(cor_path) and os.path.exists(pval_path):
        cor = pd.read_csv(cor_path, sep=None, engine='python', index_col=0)
        pval = pd.read_csv(pval_path, sep=None, engine='python', index_col=0)
        pval = pval.apply(pd.to_numeric, errors='coerce').fillna(1.0)
        sig = ((cor.abs() > 0.3) & (pval < 0.05))
        print(f"{grp}: shape {cor.shape}, significant edges {sig.values.sum() // 2}")
    else:
        print(f"{grp}: files missing")

os.makedirs("/content/sparcc_results", exist_ok=True)
for grp in GROUPS:
    base = f"/content/fastspar/sparcc_{grp}"
    for fname in [f"cor_{grp}.csv", f"pvals_{grp}.csv"]:
        src = f"{base}/{fname}"
        if os.path.exists(src):
            shutil.copy(src, "/content/sparcc_results/")

shutil.make_archive("/content/sparcc_final_pod", "zip", "/content/sparcc_results")
print("\nSaved: /content/sparcc_final_pod.zip")

VERIFICATION:
IBD: shape (95, 95), significant edges 148
rCDI: shape (158, 158), significant edges 1206
MetS: shape (179, 179), significant edges 407
MDR: shape (111, 111), significant edges 248
ICI: shape (115, 115), significant edges 726

Saved: /content/sparcc_final_pod.zip
